# Label Analyzer - CLP Compliance Checker

Production-ready analyzer using **Gemini 3 Pro** for product label compliance.

## Two-Layer Validation

**Layer 1: Gemini MEASURES**
- Font size (pixels → mm)
- Line spacing (baseline-to-baseline)
- Background/text color & contrast
- Measurement confidence (0-1)

**Layer 2: Local Deterministic Rules (100% reproducible)**
- Font size ≥ 1.2-1.8mm (depends on package size)
- Line distance ≥ 120% of font size
- White background + black text

## Output

✓ Compliance verdict (PASS/FAIL)
✓ Human review flags (uncertain or borderline)
✓ Structured JSON for portfolio updates

## Setup

In [ ]:
# Install dependencies
!pip install -q google-genai pillow pymupdf pydantic

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from label_analyzer_production import (
    LabelAnalyzer,
    PartClassification,
    image_to_base64,
    pdf_to_image
)
from PIL import Image as PIL_Image
import os
import json

print("✅ Imports OK")

In [ ]:
PROJECT_ID = "your-gcp-project-id"  # UPDATE THIS
DPI = 300
os.makedirs("data/in", exist_ok=True)
os.makedirs("data/out", exist_ok=True)

print(f"Project: {PROJECT_ID}")
print(f"DPI: {DPI}")

## Load Label

In [ ]:
IMAGE_PATH = "data/in/your_label.pdf"  # CHANGE THIS

if IMAGE_PATH.lower().endswith(".pdf"):
    img = pdf_to_image(IMAGE_PATH, dpi=DPI)
else:
    img = PIL_Image.open(IMAGE_PATH)

print(f"✅ Loaded: {img.size[0]}x{img.size[1]} px")
display(img.resize((img.width // 3, img.height // 3)))

## Authenticate & Analyze

In [ ]:
!gcloud auth application-default login

In [ ]:
analyzer = LabelAnalyzer(project_id=PROJECT_ID, dpi=DPI)
image_data = image_to_base64(img)
print("⏳ Analyzing (30-60 seconds)...")

parts = analyzer.analyze(img, image_data)

print(f"✅ Done: {len(parts)} regions detected")

## Results

In [ ]:
print("
📊 CLP COMPLIANCE RESULTS
" + "="*80)

clp_parts = [p for p in parts if p.classification == PartClassification.CLP]
compliant = [p for p in clp_parts if p.is_compliant()]
review_flagged = [p for p in clp_parts if p.needs_human_review()]

print(f"Total CLP regions: {len(clp_parts)}")
print(f"✓ Compliant: {len(compliant)}")
print(f"✗ Non-compliant: {len(clp_parts) - len(compliant)}")
print(f"⚠️  Needs review: {len(review_flagged)}")

for part in clp_parts:
    status = "PASS" if part.is_compliant() else "FAIL"
    review = " (REVIEW)" if part.needs_human_review() else ""
    conf = part.compliance_check.get("measurement_confidence", 0) if part.compliance_check else 0
    print(f"  {part.label}: {status}{review} ({conf:.0%})")


In [ ]:
for part in clp_parts:
    if not part.compliance_check:
        continue
    
    print(f"
{part.label}")
    print("-" * 60)
    
    m = part.compliance_check.get("measurements", {})
    print(f"Font: {m.get("font_size_mm", 0):.2f}mm | Line: {m.get("line_distance_mm", 0):.2f}mm")
    print(f"Confidence: {m.get("measurement_confidence", 0):.0%}")
    
    r = part.compliance_check.get("rule_results", {})
    r1 = r.get("rule_1_font_size", {})
    r2 = r.get("rule_2_line_distance", {})
    r3 = r.get("rule_3_background_contrast", {})
    
    print(f"  Rule 1 (Font): {r1.get("status")}")
    print(f"  Rule 2 (Line): {r2.get("status")}")
    print(f"  Rule 3 (Contrast): {r3.get("status")}")
    print(f"  → {part.compliance_check.get("overall_compliance")}")


## Export Results

In [ ]:
from datetime import datetime

output = {
    "timestamp": datetime.now().isoformat(),
    "image": IMAGE_PATH,
    "regions": []
}

for part in parts:
    output["regions"].append({
        "type": part.classification.value,
        "label": part.label,
        "compliant": part.is_compliant(),
        "needs_review": part.needs_human_review(),
        "compliance": part.compliance_check
    })

out_path = "data/out/report.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"✅ Saved: {out_path}")